# Residential Units ETL — QA & Summary Analysis

Reads `Parcel_Development_History` and QA tables from `C:\GIS\ParcelHistory.gdb`  
to validate ETL results and summarize unit counts by geography.

**Run order:** run the ETL (`main.py`) before opening this notebook.  
**Kernel:** ArcGIS Pro Python (`arcgispro-py3`)

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\mbindl\Documents\GitHub\Reporting\parcel_development_history_etl')

import arcpy
import pandas as pd
import numpy as np
from datetime import date

from config import (
    OUTPUT_FC, CSV_YEARS, FC_APN, FC_YEAR, FC_UNITS, FC_COUNTY,
    GDB, SPATIAL_FIELDS,
    QA_UNITS_BY_YEAR, QA_LOST_APNS, QA_SPATIAL_COMPLETENESS,
    FC_TOURIST_UNITS, FC_COMMERCIAL_SQFT,
    TOURIST_UNITS_CSV, COMMERCIAL_SQFT_CSV,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.1f}'.format)

# ── Load feature class ────────────────────────────────────────────────────────
GEO_FIELDS = [
    'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER',
    'TAZ', 'PLAN_ID', 'PLAN_NAME',
    'ZONING_ID', 'ZONING_DESCRIPTION', 'REGIONAL_LANDUSE',
    'WITHIN_TRPA_BNDY', 'WITHIN_BONUSUNIT_BNDY',
    'PARCEL_ACRES',
]
READ_FIELDS = [FC_APN, FC_YEAR, FC_UNITS, FC_COUNTY,
               FC_TOURIST_UNITS, FC_COMMERCIAL_SQFT] + GEO_FIELDS

existing = {f.name for f in arcpy.ListFields(OUTPUT_FC)}
missing  = [f for f in READ_FIELDS if f not in existing]
if missing:
    print(f'NOTE — fields not yet in FC (skipped): {missing}')
read = [f for f in READ_FIELDS if f in existing]

yr_list = ', '.join(str(y) for y in CSV_YEARS)
rows = []
with arcpy.da.SearchCursor(OUTPUT_FC, read, f"{FC_YEAR} IN ({yr_list})") as cur:
    for row in cur:
        rows.append(dict(zip(read, row)))

df = pd.DataFrame(rows).rename(columns={
    FC_APN: 'APN', FC_YEAR: 'Year', FC_UNITS: 'Units', FC_COUNTY: 'County',
    FC_TOURIST_UNITS: 'Tourist_Units', FC_COMMERCIAL_SQFT: 'Commercial_SqFt',
})
df['Units'] = df['Units'].fillna(0).astype(int)
if 'Tourist_Units' in df.columns:
    df['Tourist_Units'] = df['Tourist_Units'].fillna(0).astype(int)

# ── Load QA tables ────────────────────────────────────────────────────────────
def _read_qa(table_path, fields):
    if not arcpy.Exists(table_path):
        return pd.DataFrame()
    rows = []
    with arcpy.da.SearchCursor(table_path, fields) as cur:
        for r in cur:
            rows.append(dict(zip(fields, r)))
    return pd.DataFrame(rows)

df_yr  = _read_qa(QA_UNITS_BY_YEAR,    ['Year','CSV_Total','FC_Total','Diff','Status'])
df_sp  = _read_qa(QA_SPATIAL_COMPLETENESS, ['Field','Null_Count','Pct_Null','TRPA_Rows','Status'])
df_lost = _read_qa(QA_LOST_APNS,
                   ['APN','COUNTY','Years_Lost','Num_Years_Lost',
                    'Total_Units_CSV','Years_In_FC','Issue_Category','Suggested_Action'])

print(f'FC rows loaded  : {len(df):,}')
print(f'Unique APNs     : {df["APN"].nunique():,}')
print(f'Years           : {sorted(df["Year"].unique())}')
print('Setup complete')

---
## 0. Executive Summary

In [ ]:
latest_yr   = df['Year'].max()
earliest_yr = df['Year'].min()

print('=' * 60)
print('  RESIDENTIAL UNITS — EXECUTIVE SUMMARY')
print(f'  Generated: {date.today().strftime("%B %d, %Y").replace(" 0", " ")}')
print('=' * 60)
print(f'  Time series : {earliest_yr} – {latest_yr}')
print(f'  Unique APNs : {df["APN"].nunique():,}')
print()

# Units by year with inline bar chart
yr_totals = df.groupby('Year')['Units'].sum()
print(f'  {"Year":<6}  {"Total Units":>12}  Trend')
print(f'  {"-"*6}  {"-"*12}  {"-"*30}')
max_u = yr_totals.max()
for yr, u in yr_totals.items():
    bar   = '▓' * int(u / max_u * 30)
    match = ''
    if not df_yr.empty:
        row = df_yr[df_yr['Year'] == yr]
        if not row.empty:
            match = ' ✓' if row.iloc[0]['Diff'] == 0 else f' ✗ diff={int(row.iloc[0]["Diff"]):+,}'
    print(f'  {yr:<6}  {u:>12,}  {bar}{match}')
print()

# TRPA boundary split — latest year
if 'WITHIN_TRPA_BNDY' in df.columns:
    df_lat  = df[df['Year'] == latest_yr]
    in_trpa = int(df_lat[df_lat['WITHIN_TRPA_BNDY'] == 1]['Units'].sum())
    out_trpa= int(df_lat[df_lat['WITHIN_TRPA_BNDY'] == 0]['Units'].sum())
    null_trpa = int(df_lat[df_lat['WITHIN_TRPA_BNDY'].isna()]['Units'].sum())
    total_lat = int(df_lat['Units'].sum())
    print(f'  Units in {latest_yr}:')
    print(f'    Within TRPA boundary  : {in_trpa:>7,}  ({100*in_trpa/total_lat:.1f}%)')
    print(f'    Outside TRPA boundary : {out_trpa:>7,}  ({100*out_trpa/total_lat:.1f}%)')
    if null_trpa:
        print(f'    Not attributed (null) : {null_trpa:>7,}  ← run Step 5')
else:
    print('  WITHIN_TRPA_BNDY not populated — run Step 5 first')

print()

# Spatial attribution status
if not df_sp.empty:
    warn_fields = df_sp[df_sp['Pct_Null'] > 5]['Field'].tolist()
    if warn_fields:
        print(f'  ⚠  Spatial fields with >5% nulls: {warn_fields}')
        print('     Re-run Step 5 to populate.')
    else:
        print('  Spatial attribution: all fields <5% null ✓')
else:
    print('  Spatial attribution: QA_Spatial_Completeness not found — run ETL first')

# CSV reconciliation status
if not df_yr.empty:
    n_mismatch = int((df_yr['Diff'] != 0).sum())
    if n_mismatch == 0:
        print('  CSV ↔ FC totals: all years match ✓')
    else:
        print(f'  CSV ↔ FC totals: {n_mismatch} year(s) with discrepancies — see Section 2')

# Lost APNs
if not df_lost.empty:
    lost_units = int(df_lost['Total_Units_CSV'].sum())
    print(f'  Lost APNs: {len(df_lost):,} APNs  ({lost_units:,} units) — see Section 13')

---
## 1. CSV vs FC Reconciliation — Units by Year

The ETL writes `QA_Units_By_Year` during Step 6.  
Every row should show `Diff = 0`.  A non-zero diff means units were  
lost or added relative to the source CSV for that year.

In [ ]:
if df_yr.empty:
    print('QA_Units_By_Year not found — run ETL Step 6 first')
else:
    df_yr_disp = df_yr.sort_values('Year').copy()
    df_yr_disp['Match'] = df_yr_disp['Diff'].apply(lambda d: '✓' if d == 0 else f'✗ {int(d):+,}')
    print(df_yr_disp[['Year','CSV_Total','FC_Total','Diff','Status','Match']].to_string(index=False))
    print()
    print(f'Years matching   : {int((df_yr_disp["Diff"]==0).sum())} / {len(df_yr_disp)}')
    print(f'Total CSV units  : {int(df_yr_disp["CSV_Total"].sum()):,}')
    print(f'Total FC units   : {int(df_yr_disp["FC_Total"].sum()):,}')

---
## 2. Units by County & Year

In [ ]:
if 'County' in df.columns and df['County'].notna().any():
    piv = (df.groupby(['County','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv['TOTAL'] = piv.sum(axis=1)
    print(piv.sort_values('TOTAL', ascending=False))
else:
    print('County field not populated — run Step 1c first')

---
## 3. Units by Town Center & Year

TRPA boundary rows only.  `TOWN_CENTER` is null for parcels outside any town center  
(the catch-all "Outside Town Center" polygon is stored as null — see Step 5).

In [ ]:
if 'TOWN_CENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['TC'] = df_trpa['TOWN_CENTER'].fillna('No Town Center')
    piv = (df_trpa.groupby(['TC','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv['TOTAL'] = piv.sum(axis=1)
    print(piv.sort_values('TOTAL', ascending=False))
    print(f'\nTRPA rows: {len(df_trpa):,}  |  Town center rows: {int((df_trpa["TC"] != "No Town Center").sum()):,}')
else:
    print('TOWN_CENTER or WITHIN_TRPA_BNDY not populated — run Step 5 first')

---
## 4. Units by Location to Town Center & Year

Covers all parcels within the TRPA boundary, categorized by proximity to town centers.

In [ ]:
if 'LOCATION_TO_TOWNCENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['Loc'] = df_trpa['LOCATION_TO_TOWNCENTER'].fillna('Unknown')
    piv = (df_trpa.groupby(['Loc','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv['TOTAL'] = piv.sum(axis=1)
    print(piv.sort_values('TOTAL', ascending=False))
else:
    print('LOCATION_TO_TOWNCENTER or WITHIN_TRPA_BNDY not populated — run Step 5 first')

---
## 5. Units by Regional Land Use & Year

TRPA boundary rows only.

In [ ]:
if 'REGIONAL_LANDUSE' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['RLU'] = df_trpa['REGIONAL_LANDUSE'].fillna('Unknown')
    piv = (df_trpa.groupby(['RLU','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv['TOTAL'] = piv.sum(axis=1)
    print(piv.sort_values('TOTAL', ascending=False))
else:
    print('REGIONAL_LANDUSE or WITHIN_TRPA_BNDY not populated — run Step 5 first')

---
## 6. Units by TAZ & Year

Traffic Analysis Zones — TRPA boundary rows only.  Top 20 by total units shown.

In [ ]:
if 'TAZ' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[(df['WITHIN_TRPA_BNDY'] == 1) & df['TAZ'].notna()]
    piv = (df_trpa.groupby(['TAZ','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv['TOTAL'] = piv.sum(axis=1)
    piv = piv.sort_values('TOTAL', ascending=False)
    print(f'TAZ count: {len(piv)}  (showing top 20)')
    print(piv.head(20))
else:
    print('TAZ or WITHIN_TRPA_BNDY not populated — run Step 5 first')

---
## 7. Units by Zoning & Year

TRPA boundary rows only.

In [ ]:
if 'ZONING_DESCRIPTION' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['Zone'] = df_trpa['ZONING_DESCRIPTION'].fillna('Unknown')
    piv = (df_trpa.groupby(['Zone','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv['TOTAL'] = piv.sum(axis=1)
    piv = piv.sort_values('TOTAL', ascending=False)
    pct_unknown = 0
    if 'Unknown' in piv.index:
        pct_unknown = 100 * piv.loc['Unknown','TOTAL'] / piv['TOTAL'].sum()
    print(f'Zoning categories: {len(piv)}  (Unknown: {pct_unknown:.1f}% of units)')
    print(piv)
else:
    print('ZONING_DESCRIPTION or WITHIN_TRPA_BNDY not populated — run Step 5 first')

---
## 8. Spatial Attribution Completeness

Shows null rates per spatial field for TRPA boundary rows.  
Written by Step 6 QA.  All fields should be `OK` (< 5% null) after Step 5 runs.

If fields show `WARN` here, re-run Step 5 — the service cache likely failed for that layer.

In [ ]:
if df_sp.empty:
    print('QA_Spatial_Completeness not found — run ETL Step 6 first')
else:
    df_sp_disp = df_sp.sort_values('Pct_Null', ascending=False).copy()
    print(df_sp_disp.to_string(index=False))
    print()
    warn = df_sp_disp[df_sp_disp['Pct_Null'] > 5]
    if len(warn):
        print(f'Fields with >5% nulls ({len(warn)}):')
        for _, r in warn.iterrows():
            print(f'  {r["Field"]:<35} {r["Pct_Null"]:>6.1f}% null  ({int(r["Null_Count"]):,} rows)')
    else:
        print('All fields within acceptable null rate (<5%) ✓')

---
## 9. Bonus Unit Area — TRPA Boundary

Shows how many residential units fall within the Bonus Unit boundary each year.

In [ ]:
if 'WITHIN_BONUSUNIT_BNDY' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1]
    piv = (df_trpa.groupby(['WITHIN_BONUSUNIT_BNDY','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv.index = piv.index.map({0: 'Outside Bonus Area', 1: 'Within Bonus Area'})
    piv['TOTAL'] = piv.sum(axis=1)
    print(piv)
else:
    print('WITHIN_BONUSUNIT_BNDY or WITHIN_TRPA_BNDY not populated — run Step 5 first')

---
## 10. Year-over-Year Change by Town Center

In [ ]:
if 'TOWN_CENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['TC'] = df_trpa['TOWN_CENTER'].fillna('No Town Center')
    piv = (df_trpa.groupby(['TC','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    years = sorted(CSV_YEARS)
    chg = pd.DataFrame(index=piv.index)
    for i in range(1, len(years)):
        y0, y1 = years[i-1], years[i]
        if y0 in piv.columns and y1 in piv.columns:
            chg[f'{y0}→{y1}'] = piv[y1] - piv[y0]
    chg['NET (2012→latest)'] = piv.get(years[-1], 0) - piv.get(years[0], 0)
    print(chg.sort_values('NET (2012→latest)', ascending=False))
else:
    print('TOWN_CENTER or WITHIN_TRPA_BNDY not populated — run Step 5 first')

---
## 11. Parcel Count vs Unit Count by Year

Sanity check: the number of parcels with units and the average units per developed parcel  
should be stable over time.  Large swings may indicate data issues.

In [ ]:
scope = df[df['WITHIN_TRPA_BNDY'] == 1] if 'WITHIN_TRPA_BNDY' in df.columns else df

density = (scope.groupby('Year')
             .agg(
                 Parcels          = ('APN',   'count'),
                 Parcels_w_Units  = ('Units', lambda x: (x > 0).sum()),
                 Total_Units      = ('Units', 'sum'),
             )
             .reset_index())
density['Pct_Developed'] = (density['Parcels_w_Units'] / density['Parcels'] * 100).round(1)
density['Avg_Units']     = (density['Total_Units'] / density['Parcels_w_Units'].replace(0, np.nan)).round(2)
print(density.to_string(index=False))

---
## 12. Dropout Flag — Units Temporarily Zero

APNs where units are > 0 in year N-1, drop to 0 in year N, then recover in year N+1.  
These are likely data entry gaps, not real demolitions.

In [ ]:
pivot = df.pivot_table(index='APN', columns='Year', values='Units', fill_value=0)
years = sorted(pivot.columns)
dropout_rows = []

for i, yr in enumerate(years[1:-1]):
    prev_yr = years[i]
    next_yr = years[i + 2]
    mask = (pivot[prev_yr] > 0) & (pivot[yr] == 0) & (pivot[next_yr] > 0)
    for apn in pivot.index[mask]:
        dropout_rows.append({
            'APN': apn,
            'Gap_Year': yr,
            'Before': int(pivot.loc[apn, prev_yr]),
            'After':  int(pivot.loc[apn, next_yr]),
        })

df_dropout = pd.DataFrame(dropout_rows)
if df_dropout.empty:
    print('No dropout gaps detected ✓')
else:
    print(f'Dropout gaps: {len(df_dropout):,} APN-year pairs  ({df_dropout["APN"].nunique():,} unique APNs)')
    gap_yr_counts = df_dropout.groupby('Gap_Year').size().rename('Count')
    print('\nBy gap year:')
    print(gap_yr_counts.to_string())
    print('\nTop 20 APNs with most gaps:')
    print(df_dropout.groupby('APN').size().nlargest(20).rename('Gaps').to_string())

---
## 13. Lost APNs Summary

APNs with units in the source CSV but 0 or missing in the FC.  
Written by Step 6 QA.

In [ ]:
if df_lost.empty:
    print('QA_Lost_APNs not found — run ETL first')
else:
    print(f'Total lost APNs  : {len(df_lost):,}')
    print(f'Total lost units : {int(df_lost["Total_Units_CSV"].sum()):,}')
    print()
    cat_sum = (df_lost.groupby('Issue_Category')
                 .agg(APNs=('APN','count'), Units=('Total_Units_CSV','sum'))
                 .sort_values('Units', ascending=False))
    print(cat_sum)

In [ ]:
if not df_lost.empty:
    pd.set_option('display.max_colwidth', 80)
    top = df_lost.nlargest(20, 'Total_Units_CSV')[[
        'APN','COUNTY','Issue_Category','Total_Units_CSV','Years_Lost','Suggested_Action'
    ]]
    print('Top 20 lost APNs by units:')
    print(top.to_string(index=False))

---
## 14. Commercial Floor Area — CSV vs FC Reconciliation

Compares `CommercialFloorArea_2012to2025.csv` totals against `CommercialFloorArea_SqFt`  
in the FC by year.  Every row should show `Diff = 0`.

In [ ]:
if 'Commercial_SqFt' not in df.columns:
    print('Commercial_SqFt not in FC — run ETL s04b first')
else:
    cfa_csv = pd.read_csv(COMMERCIAL_SQFT_CSV)
    apn_col = cfa_csv.columns[0]
    year_cols = [c for c in cfa_csv.columns if c.startswith('CY')]

    cfa_long = cfa_csv.melt(id_vars=[apn_col], value_vars=year_cols,
                             var_name='YearStr', value_name='CSV_SqFt')
    cfa_long['Year']    = cfa_long['YearStr'].str.replace('CY', '').astype(int)
    cfa_long['CSV_SqFt'] = pd.to_numeric(cfa_long['CSV_SqFt'], errors='coerce').fillna(0)
    csv_cfa = cfa_long.groupby('Year')['CSV_SqFt'].sum().reset_index()
    csv_cfa.columns = ['Year', 'CSV_Total']

    fc_cfa = (df.groupby('Year')['Commercial_SqFt']
                .sum()
                .reset_index()
                .rename(columns={'Commercial_SqFt': 'FC_Total'}))

    cfa_qa = csv_cfa.merge(fc_cfa, on='Year', how='outer').fillna(0).sort_values('Year')
    cfa_qa['Diff']  = cfa_qa['FC_Total'] - cfa_qa['CSV_Total']
    cfa_qa['Match'] = cfa_qa['Diff'].apply(lambda d: '✓' if abs(d) < 0.5 else f'✗ {d:+,.0f}')

    print('Commercial Floor Area (SqFt) — CSV vs FC')
    print(cfa_qa[['Year', 'CSV_Total', 'FC_Total', 'Diff', 'Match']].to_string(index=False))
    print()
    n_mismatch = int((cfa_qa['Diff'].abs() >= 0.5).sum())
    if n_mismatch == 0:
        print(f'All {len(cfa_qa)} years match ✓')
        print(f'Total CSV SqFt : {cfa_qa["CSV_Total"].sum():,.0f}')
        print(f'Total FC  SqFt : {cfa_qa["FC_Total"].sum():,.0f}')
    else:
        bad = cfa_qa[cfa_qa['Diff'].abs() >= 0.5]
        print(f'{n_mismatch} year(s) with discrepancies ✗')
        print(bad[['Year', 'CSV_Total', 'FC_Total', 'Diff']].to_string(index=False))

---
## 15. Tourist Accommodation Units — CSV vs FC Reconciliation

Compares `TouristUnits_2012to2025.csv` totals against `TouristAccommodation_Units`  
in the FC by year.  Every row should show `Diff = 0`.

In [ ]:
if 'Tourist_Units' not in df.columns:
    print('Tourist_Units not in FC — run ETL s04b first')
else:
    tau_csv = pd.read_csv(TOURIST_UNITS_CSV)
    apn_col = tau_csv.columns[0]
    year_cols = [c for c in tau_csv.columns if c.startswith('CY')]

    tau_long = tau_csv.melt(id_vars=[apn_col], value_vars=year_cols,
                             var_name='YearStr', value_name='CSV_Units')
    tau_long['Year']      = tau_long['YearStr'].str.replace('CY', '').astype(int)
    tau_long['CSV_Units'] = pd.to_numeric(tau_long['CSV_Units'], errors='coerce').fillna(0)
    csv_tau = tau_long.groupby('Year')['CSV_Units'].sum().reset_index()
    csv_tau.columns = ['Year', 'CSV_Total']

    fc_tau = (df.groupby('Year')['Tourist_Units']
                .sum()
                .reset_index()
                .rename(columns={'Tourist_Units': 'FC_Total'}))

    tau_qa = csv_tau.merge(fc_tau, on='Year', how='outer').fillna(0).sort_values('Year')
    tau_qa['Diff']  = tau_qa['FC_Total'] - tau_qa['CSV_Total']
    tau_qa['Match'] = tau_qa['Diff'].apply(lambda d: '✓' if d == 0 else f'✗ {int(d):+,}')

    print('Tourist Accommodation Units — CSV vs FC')
    print(tau_qa[['Year', 'CSV_Total', 'FC_Total', 'Diff', 'Match']].to_string(index=False))
    print()
    n_mismatch = int((tau_qa['Diff'] != 0).sum())
    if n_mismatch == 0:
        print(f'All {len(tau_qa)} years match ✓')
        print(f'Total CSV units : {int(tau_qa["CSV_Total"].sum()):,}')
        print(f'Total FC  units : {int(tau_qa["FC_Total"].sum()):,}')
    else:
        bad = tau_qa[tau_qa['Diff'] != 0]
        print(f'{n_mismatch} year(s) with discrepancies ✗')
        print(bad[['Year', 'CSV_Total', 'FC_Total', 'Diff']].to_string(index=False))